In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam
import shutil
import time
import random
import matplotlib.patches as patches

In [ ]:
CLASS_NAMES = ["picaras", "chokosoda", "oreo"]

# Cargar modelos:

###  Funciones de perdida

In [ ]:
# ── 1. FUNCIONES DE PÉRDIDA modelo 1──────────────────────────────

def iou_metric(y_true, y_pred):
    """Métrica IoU para monitorear en TensorBoard (no diferenciable, solo lectura)."""
    x1g, y1g, x2g, y2g = tf.unstack(y_true, axis=-1)
    x1p, y1p, x2p, y2p = tf.unstack(y_pred, axis=-1)
    xi1 = tf.maximum(x1g, x1p);  yi1 = tf.maximum(y1g, y1p)
    xi2 = tf.minimum(x2g, x2p);  yi2 = tf.minimum(y2g, y2p)
    inter = tf.maximum(xi2 - xi1, 0.0) * tf.maximum(yi2 - yi1, 0.0)
    area_g = tf.maximum((x2g - x1g) * (y2g - y1g), 0.0)
    area_p = tf.maximum((x2p - x1p) * (y2p - y1p), 0.0)
    union  = area_g + area_p - inter + 1e-7
    return tf.reduce_mean(inter / union)


def giou_loss(y_true, y_pred):
    """
    GIoU con protección contra coordenadas invertidas.
    Las predicciones se ordenan (x1<x2, y1<y2) antes de calcular.
    """
    # --- Garantizar x1<x2, y1<y2 en predicciones ---
    x1p_raw, y1p_raw, x2p_raw, y2p_raw = tf.unstack(y_pred, axis=-1)
    x1p = tf.minimum(x1p_raw, x2p_raw)
    y1p = tf.minimum(y1p_raw, y2p_raw)
    x2p = tf.maximum(x1p_raw, x2p_raw)
    y2p = tf.maximum(y1p_raw, y2p_raw)

    x1g, y1g, x2g, y2g = tf.unstack(y_true, axis=-1)

    # Intersección
    xi1 = tf.maximum(x1g, x1p);  yi1 = tf.maximum(y1g, y1p)
    xi2 = tf.minimum(x2g, x2p);  yi2 = tf.minimum(y2g, y2p)
    inter = tf.maximum(xi2 - xi1, 0.0) * tf.maximum(yi2 - yi1, 0.0)

    area_g = (x2g - x1g) * (y2g - y1g)
    area_p = (x2p - x1p) * (y2p - y1p)
    union  = area_g + area_p - inter + 1e-7
    iou    = inter / union

    # Caja envolvente (penalización GIoU)
    xc1 = tf.minimum(x1g, x1p);  yc1 = tf.minimum(y1g, y1p)
    xc2 = tf.maximum(x2g, x2p);  yc2 = tf.maximum(y2g, y2p)
    area_c = (xc2 - xc1) * (yc2 - yc1) + 1e-7

    giou = iou - (area_c - union) / area_c
    return tf.reduce_mean(1.0 - giou)


def mse_loss(y_true, y_pred):
    """MSE puro como etapa de calentamiento."""
    return tf.reduce_mean(tf.square(y_true - y_pred))


def bbox_combined_loss(y_true, y_pred):
    """
    MSE + GIoU con peso equilibrado.
    MSE da gradientes suaves desde el inicio;
    GIoU refina la superposición espacial.
    """
    return mse_loss(y_true, y_pred) + giou_loss(y_true, y_pred)


# ── 1. FUNCIONES DE PÉRDIDA modelo 2──────────────────────────────

def iou_metric_2(y_true, y_pred):
    """Métrica IoU para monitorear en TensorBoard (no diferenciable, solo lectura)."""
    x1g, y1g, x2g, y2g = tf.unstack(y_true, axis=-1)
    x1p, y1p, x2p, y2p = tf.unstack(y_pred, axis=-1)
    xi1 = tf.maximum(x1g, x1p);  yi1 = tf.maximum(y1g, y1p)
    xi2 = tf.minimum(x2g, x2p);  yi2 = tf.minimum(y2g, y2p)
    inter = tf.maximum(xi2 - xi1, 0.0) * tf.maximum(yi2 - yi1, 0.0)
    area_g = tf.maximum((x2g - x1g) * (y2g - y1g), 0.0)
    area_p = tf.maximum((x2p - x1p) * (y2p - y1p), 0.0)
    union  = area_g + area_p - inter + 1e-7
    return tf.reduce_mean(inter / union)


def giou_loss_2(y_true, y_pred):
    """
    GIoU con protección contra coordenadas invertidas.
    Las predicciones se ordenan (x1<x2, y1<y2) antes de calcular.
    """
    # --- Garantizar x1<x2, y1<y2 en predicciones ---
    x1p_raw, y1p_raw, x2p_raw, y2p_raw = tf.unstack(y_pred, axis=-1)
    x1p = tf.minimum(x1p_raw, x2p_raw)
    y1p = tf.minimum(y1p_raw, y2p_raw)
    x2p = tf.maximum(x1p_raw, x2p_raw)
    y2p = tf.maximum(y1p_raw, y2p_raw)

    x1g, y1g, x2g, y2g = tf.unstack(y_true, axis=-1)

    # Intersección
    xi1 = tf.maximum(x1g, x1p);  yi1 = tf.maximum(y1g, y1p)
    xi2 = tf.minimum(x2g, x2p);  yi2 = tf.minimum(y2g, y2p)
    inter = tf.maximum(xi2 - xi1, 0.0) * tf.maximum(yi2 - yi1, 0.0)

    area_g = (x2g - x1g) * (y2g - y1g)
    area_p = (x2p - x1p) * (y2p - y1p)
    union  = area_g + area_p - inter + 1e-7
    iou    = inter / union

    # Caja envolvente (penalización GIoU)
    xc1 = tf.minimum(x1g, x1p);  yc1 = tf.minimum(y1g, y1p)
    xc2 = tf.maximum(x2g, x2p);  yc2 = tf.maximum(y2g, y2p)
    area_c = (xc2 - xc1) * (yc2 - yc1) + 1e-7

    giou = iou - (area_c - union) / area_c
    return tf.reduce_mean(1.0 - giou)


def mse_loss_2(y_true, y_pred):
    """MSE puro como etapa de calentamiento."""
    return tf.reduce_mean(tf.square(y_true - y_pred))


def bbox_combined_loss_2(y_true, y_pred):
    mse  = mse_loss_2(y_true, y_pred)
    giou = giou_loss_2(y_true, y_pred)

    # Penalizar si el área predicha es muy diferente al área real
    w_true = y_true[:, 2] - y_true[:, 0]
    h_true = y_true[:, 3] - y_true[:, 1]
    w_pred = y_pred[:, 2] - y_pred[:, 0]
    h_pred = y_pred[:, 3] - y_pred[:, 1]
    size_loss = tf.reduce_mean(tf.square(w_true - w_pred) + tf.square(h_true - h_pred))

    return mse + giou + 0.5 * size_loss




### Cargar:

In [ ]:

# ── 1. Cargar el modelo 1──────────────────────────────────
modelo1 = tf.keras.models.load_model(
    'mejor_modelo_mobilenet.keras',
    custom_objects={
        'bbox_combined_loss': bbox_combined_loss,
        'mse_loss': mse_loss,
        'giou_loss': giou_loss,
        'iou_metric': iou_metric
    }
)

# ── 1. Cargar el modelo 2 ──────────────────────────────────
modelo2 = tf.keras.models.load_model(
    'mejor_modelo_mobilenet_rotacion.keras',
    custom_objects={
        'bbox_combined_loss': bbox_combined_loss_2,
        'mse_loss': mse_loss_2,
        'giou_loss': giou_loss_2,
        'iou_metric': iou_metric_2
    }
)




# New

In [ ]:
# ── Predicción combinada ─────────────────────────────────
def predict_combinado(input_batch):
    """
    Usa clasificación del modelo1 y bbox del modelo2.
    """
    _, class_pred  = modelo1.predict(input_batch, verbose=0)  # solo nos interesa la clase
    bbox_pred, _   = modelo2.predict(input_batch, verbose=0)  # solo nos interesa el bbox

    # Ordenar coordenadas
    x1 = np.minimum(bbox_pred[:, 0], bbox_pred[:, 2])
    y1 = np.minimum(bbox_pred[:, 1], bbox_pred[:, 3])
    x2 = np.maximum(bbox_pred[:, 0], bbox_pred[:, 2])
    y2 = np.maximum(bbox_pred[:, 1], bbox_pred[:, 3])
    bboxes = np.stack([x1, y1, x2, y2], axis=-1)

    clase     = CLASS_NAMES[np.argmax(class_pred[0])]
    confianza = np.max(class_pred[0])

    return bboxes[0], clase, confianza

In [ ]:
def real_time_detection(camera_index=0, confidence_threshold=0.8):
    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened():
        print(f"No se pudo abrir la cámara con índice {camera_index}")
        return

    print("Presiona 'q' para salir de la detección en tiempo real")
    print("Iniciando detección...")

    prev_time = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            print("No se pudo leer el frame de la cámara")
            break

        original_h, original_w = frame.shape[:2]

        # Preprocesar
        img_resized = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
        input_batch = np.expand_dims(img_resized / 255.0, axis=0)

        # ── Predicción combinada ─────────────────────────
        bbox, clase, confianza = predict_combinado(input_batch)

        if confianza >= confidence_threshold:
            x1 = int(bbox[0] * original_w)
            y1 = int(bbox[1] * original_h)
            x2 = int(bbox[2] * original_w)
            y2 = int(bbox[3] * original_h)

            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame, f"{clase}: {confianza:.2f}", (x1, max(y1 - 10, 20)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        # FPS
        curr_time = time.time()
        fps       = 1 / (curr_time - prev_time) if prev_time > 0 else 0
        prev_time = curr_time
        cv2.putText(frame, f"FPS: {fps:.1f}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)

        cv2.imshow('Detección en Tiempo Real', frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

In [ ]:
# Ejecutar detección en tiempo real
real_time_detection(confidence_threshold=0.95)